In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("Data_set_prep_assignment_1.csv")
TARGET = "AnimalId"
FEATURES = [
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
]
K = 5
MAX_MODEL_ROWS = 100_000
CHUNK_SIZE = 250_000

rng = np.random.default_rng(42)
model_data = pd.DataFrame()

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[TARGET, *FEATURES],
    dtype={TARGET: "string"},
    chunksize=CHUNK_SIZE,
):
    for feature in FEATURES:
        chunk[feature] = pd.to_numeric(chunk[feature], errors="coerce")

    chunk = chunk.dropna(subset=[TARGET, *FEATURES])
    if chunk.empty:
        continue

    chunk["_sample_key"] = rng.random(len(chunk))
    model_data = pd.concat([model_data, chunk], ignore_index=True)
    if len(model_data) > MAX_MODEL_ROWS:
        model_data = model_data.nsmallest(MAX_MODEL_ROWS, "_sample_key")

model_data = model_data.drop(columns="_sample_key").reset_index(drop=True)
if len(model_data) <= K:
    raise ValueError(f"Need more than {K} labeled rows; found {len(model_data)}.")

X = model_data[FEATURES]
y = model_data[TARGET].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=K, algorithm="kd_tree", n_jobs=-1),
)
knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Unique AnimalId labels: {y.nunique():,}")
print(f"K: {K}")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")